# 📊 Automated Financial Data Extraction Tool (AI & NLP)

**Author:** Arnab Banerjee  
**GitHub:** [github.com/arnab2308](https://github.com/arnab2308)  
**LinkedIn:** [linkedin.com/in/arnabbanerjee2308](https://linkedin.com/in/arnabbanerjee2308)

---

## 🎯 Problem Statement

Private market data often comes in **unstructured PDFs**—quarterly fund reports, portfolio company financials, and partnership statements. This tool automates extraction of key financial metrics using **OCR and Large Language Models (LLMs)**, converting messy PDF reports into clean, analysis-ready data.

### Use Cases
- Private Credit Fund Reporting
- LP Transparency & Portfolio Monitoring
- Due Diligence Data Room Processing

---

## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
!pip install pdfplumber pandas openpyxl -q

# Optional: For LLM integration (uncomment as needed)
# !pip install openai anthropic -q

print("✅ Installation complete!")

In [ ]:
# Import libraries
import os
import re
import json
import logging
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd
import pdfplumber

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully!")

## 2️⃣ Core Classes & Functions

In [ ]:
@dataclass
class FinancialMetrics:
    """Data class for standardized financial metrics extraction."""
    company_name: str
    report_date: str
    report_type: str
    
    # Income Statement Metrics
    revenue: Optional[float] = None
    gross_profit: Optional[float] = None
    operating_income: Optional[float] = None
    ebitda: Optional[float] = None
    net_income: Optional[float] = None
    
    # Balance Sheet Metrics
    total_assets: Optional[float] = None
    total_liabilities: Optional[float] = None
    total_equity: Optional[float] = None
    cash_and_equivalents: Optional[float] = None
    total_debt: Optional[float] = None
    net_debt: Optional[float] = None
    
    # Key Ratios
    debt_to_equity: Optional[float] = None
    current_ratio: Optional[float] = None
    
    # Metadata
    extraction_confidence: float = 0.0
    source_file: str = ""
    extraction_timestamp: str = ""

print("✅ FinancialMetrics dataclass defined!")

In [ ]:
class LLMPromptBuilder:
    """
    Builds structured prompts for LLM-based financial data extraction.
    Designed to work with OpenAI GPT-4, Anthropic Claude, or similar models.
    """
    
    SYSTEM_PROMPT = """You are a financial data extraction specialist. Your task is to 
extract specific financial metrics from unstructured text extracted from fund reports 
and financial statements. 

Rules:
1. Extract only explicitly stated values - do not calculate or infer
2. Maintain original currency and units
3. Convert values to millions (M) for consistency
4. Return null for metrics not found in the text
5. Include confidence score (0-1) for each extraction
"""

    @staticmethod
    def build_extraction_prompt(text: str, target_metrics: List[str]) -> str:
        """Build a structured prompt for financial metric extraction."""
        metrics_list = "\n".join([f"- {m}" for m in target_metrics])
        
        prompt = f"""
### TASK: Extract Financial Metrics

### TARGET METRICS:
{metrics_list}

### SOURCE TEXT:
\"\"\"
{text[:8000]}
\"\"\"

### INSTRUCTIONS:
1. Identify each metric from the source text
2. Extract the numerical value with its unit/currency
3. Note the fiscal period if mentioned
4. Provide confidence score (0-1) based on clarity of the data

### OUTPUT FORMAT (JSON):
{{
    "metrics": {{
        "metric_name": {{
            "value": <number or null>,
            "unit": "<currency/unit>",
            "period": "<fiscal period>",
            "confidence": <0-1>,
            "source_context": "<relevant text snippet>"
        }}
    }},
    "tables_identified": <number>,
    "overall_confidence": <0-1>
}}
"""
        return prompt

print("✅ LLMPromptBuilder class defined!")

In [ ]:
class PDFTextExtractor:
    """Handles PDF text and table extraction with quality checks."""
    
    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.text_content: Dict[int, str] = {}
        self.tables: Dict[int, List] = {}
        self.metadata: Dict = {}
        
    def extract_all(self) -> Tuple[str, List[pd.DataFrame]]:
        """Extract all text and tables from PDF."""
        logger.info(f"Extracting content from: {self.pdf_path}")
        
        full_text = []
        all_tables = []
        
        with pdfplumber.open(self.pdf_path) as pdf:
            self.metadata = {"pages": len(pdf.pages), "filename": self.pdf_path.name}
            
            for i, page in enumerate(pdf.pages):
                # Extract text
                page_text = page.extract_text() or ""
                self.text_content[i] = page_text
                full_text.append(f"\n--- Page {i+1} ---\n{page_text}")
                
                # Extract tables
                tables = page.extract_tables()
                if tables:
                    self.tables[i] = tables
                    for table in tables:
                        if table and len(table) > 1:
                            try:
                                df = pd.DataFrame(table[1:], columns=table[0])
                                df['source_page'] = i + 1
                                all_tables.append(df)
                            except Exception as e:
                                logger.warning(f"Table parsing error on page {i+1}: {e}")
        
        logger.info(f"Extracted {len(full_text)} pages, {len(all_tables)} tables")
        return "\n".join(full_text), all_tables

print("✅ PDFTextExtractor class defined!")

In [ ]:
class DataValidator:
    """
    Validates extracted financial data for quality and consistency.
    Implements automated checks against historical data and logical rules.
    """
    
    VALIDATION_RULES = {
        'revenue': {'min': 0, 'max': 1e12},
        'ebitda': {'min': -1e11, 'max': 1e11},
        'net_debt': {'min': -1e11, 'max': 1e11},
        'total_assets': {'min': 0, 'max': 1e13},
        'debt_to_equity': {'min': -10, 'max': 50},
    }
    
    def __init__(self, historical_data: Optional[pd.DataFrame] = None):
        self.historical_data = historical_data
        self.validation_results: List[Dict] = []
        
    def validate(self, metrics: FinancialMetrics) -> Dict[str, Any]:
        """Run all validation checks on extracted metrics."""
        results = {
            'is_valid': True,
            'errors': [],
            'warnings': [],
            'checks_passed': 0,
            'checks_total': 0
        }
        
        metrics_dict = asdict(metrics)
        
        # Range validation
        for field, rules in self.VALIDATION_RULES.items():
            value = metrics_dict.get(field)
            if value is not None:
                results['checks_total'] += 1
                if rules['min'] <= value <= rules['max']:
                    results['checks_passed'] += 1
                else:
                    results['errors'].append(
                        f"{field}: Value {value} outside valid range [{rules['min']}, {rules['max']}]"
                    )
                    results['is_valid'] = False
        
        # Logical consistency checks
        results['checks_total'] += 3
        
        # Check: Total Assets = Total Liabilities + Total Equity
        if all(metrics_dict.get(f) is not None for f in ['total_assets', 'total_liabilities', 'total_equity']):
            balance = metrics.total_assets - (metrics.total_liabilities + metrics.total_equity)
            if abs(balance) > 0.01 * metrics.total_assets:
                results['warnings'].append(f"Balance sheet imbalance: {balance:,.0f}")
            else:
                results['checks_passed'] += 1
        
        # Check: Net Debt = Total Debt - Cash
        if all(metrics_dict.get(f) is not None for f in ['net_debt', 'total_debt', 'cash_and_equivalents']):
            expected_net_debt = metrics.total_debt - metrics.cash_and_equivalents
            if abs(metrics.net_debt - expected_net_debt) > 0.01 * abs(expected_net_debt):
                results['warnings'].append(f"Net debt inconsistency")
            else:
                results['checks_passed'] += 1
        
        # Check: EBITDA margin reasonable
        if metrics.ebitda is not None and metrics.revenue is not None and metrics.revenue > 0:
            ebitda_margin = metrics.ebitda / metrics.revenue
            if not -0.5 <= ebitda_margin <= 0.8:
                results['warnings'].append(f"Unusual EBITDA margin: {ebitda_margin:.1%}")
            else:
                results['checks_passed'] += 1
        
        return results

print("✅ DataValidator class defined!")

## 3️⃣ Demo: Extract from Sample Fund Report

Let's demonstrate the extraction pipeline using a simulated private credit fund report.

In [ ]:
# Sample Private Credit Fund Report Text
# In production, this would be extracted from PDF using PDFTextExtractor

SAMPLE_FUND_REPORT = """
BLACKROCK PRIVATE CREDIT FUND II
QUARTERLY REPORT - Q3 2024

PORTFOLIO SUMMARY
=================
As of September 30, 2024

Net Asset Value (NAV): $2,847,500,000
Total Commitments: $3,500,000,000
Capital Called: 81.4%

UNDERLYING PORTFOLIO COMPANIES - FINANCIAL HIGHLIGHTS
=====================================================

Company: TechFlow Solutions LLC
Industry: Software/SaaS
Investment Type: Senior Secured Term Loan
Principal Amount: $125,000,000

Financial Metrics (LTM September 2024):
- Revenue: $287,400,000
- Gross Profit: $201,180,000 (70% margin)
- EBITDA: $74,724,000 (26% margin)
- Net Income: $31,614,000
- Total Assets: $412,500,000
- Total Liabilities: $298,750,000
- Total Equity: $113,750,000
- Cash and Equivalents: $42,300,000
- Total Debt: $225,000,000
- Net Debt: $182,700,000

---

Company: MedSupply Holdings Inc.
Industry: Healthcare Distribution
Investment Type: Unitranche Facility
Principal Amount: $85,000,000

Financial Metrics (LTM September 2024):
- Revenue: $198,600,000
- Gross Profit: $49,650,000 (25% margin)
- EBITDA: $29,790,000 (15% margin)
- Net Income: $12,100,000
- Total Assets: $278,400,000
- Total Liabilities: $189,200,000
- Total Equity: $89,200,000
- Cash and Equivalents: $28,500,000
- Total Debt: $142,000,000
- Net Debt: $113,500,000
"""

print("📄 Sample fund report loaded!")
print(f"Report length: {len(SAMPLE_FUND_REPORT)} characters")

In [ ]:
# Step 1: Build LLM Extraction Prompt
print("="*60)
print("STEP 1: Building LLM Extraction Prompt")
print("="*60)

prompt_builder = LLMPromptBuilder()
target_metrics = ['Revenue', 'EBITDA', 'Net Debt', 'Total Assets', 'Total Liabilities', 'Total Equity']

extraction_prompt = prompt_builder.build_extraction_prompt(SAMPLE_FUND_REPORT, target_metrics)

print("\n📝 Prompt Preview (first 600 chars):")
print("-"*40)
print(extraction_prompt[:600] + "...")

In [ ]:
# Step 2: Simulated LLM Extraction Results
# In production, this would call OpenAI/Anthropic API

print("\n" + "="*60)
print("STEP 2: LLM Extraction Results")
print("="*60)

# Simulated extraction (what the LLM would return)
extracted_companies = [
    {
        'company_name': 'TechFlow Solutions LLC',
        'industry': 'Software/SaaS',
        'revenue': 287400000,
        'gross_profit': 201180000,
        'ebitda': 74724000,
        'net_income': 31614000,
        'total_assets': 412500000,
        'total_liabilities': 298750000,
        'total_equity': 113750000,
        'cash_and_equivalents': 42300000,
        'total_debt': 225000000,
        'net_debt': 182700000,
        'debt_to_equity': 1.98,
        'current_ratio': 1.8
    },
    {
        'company_name': 'MedSupply Holdings Inc.',
        'industry': 'Healthcare Distribution',
        'revenue': 198600000,
        'gross_profit': 49650000,
        'ebitda': 29790000,
        'net_income': 12100000,
        'total_assets': 278400000,
        'total_liabilities': 189200000,
        'total_equity': 89200000,
        'cash_and_equivalents': 28500000,
        'total_debt': 142000000,
        'net_debt': 113500000,
        'debt_to_equity': 2.12,
        'current_ratio': 1.4
    }
]

print("\n📊 Extracted Financial Metrics:")
print("-"*40)
for company in extracted_companies:
    print(f"\n🏢 {company['company_name']} ({company['industry']})")
    print(f"   Revenue:       ${company['revenue']:>15,}")
    print(f"   EBITDA:        ${company['ebitda']:>15,}")
    print(f"   Net Debt:      ${company['net_debt']:>15,}")
    print(f"   EBITDA Margin: {company['ebitda']/company['revenue']*100:>14.1f}%")

In [ ]:
# Step 3: Data Validation
print("\n" + "="*60)
print("STEP 3: Data Validation")
print("="*60)

validator = DataValidator()

for company in extracted_companies:
    metrics = FinancialMetrics(
        company_name=company['company_name'],
        report_date='2024-09-30',
        report_type='Quarterly Fund Report',
        revenue=company['revenue'],
        gross_profit=company.get('gross_profit'),
        ebitda=company['ebitda'],
        net_income=company.get('net_income'),
        total_assets=company['total_assets'],
        total_liabilities=company['total_liabilities'],
        total_equity=company['total_equity'],
        cash_and_equivalents=company['cash_and_equivalents'],
        total_debt=company['total_debt'],
        net_debt=company['net_debt'],
        debt_to_equity=company.get('debt_to_equity'),
        current_ratio=company.get('current_ratio'),
        source_file='sample_fund_report.pdf',
        extraction_timestamp=datetime.now().isoformat()
    )
    
    results = validator.validate(metrics)
    
    print(f"\n🏢 {company['company_name']}")
    status = '✅ PASSED' if results['is_valid'] else '❌ FAILED'
    print(f"   Validation Status: {status}")
    print(f"   Checks Passed: {results['checks_passed']}/{results['checks_total']}")
    
    # Verify balance sheet equation
    balance_check = company['total_assets'] - (company['total_liabilities'] + company['total_equity'])
    print(f"   Balance Sheet Check: Assets - (Liab + Equity) = ${balance_check:,}")
    
    # Verify net debt calculation
    net_debt_calc = company['total_debt'] - company['cash_and_equivalents']
    print(f"   Net Debt Check: Calculated=${net_debt_calc:,}, Reported=${company['net_debt']:,}")
    
    if results['warnings']:
        print(f"   ⚠️ Warnings: {results['warnings']}")

In [ ]:
# Step 4: Export to Structured Format
print("\n" + "="*60)
print("STEP 4: Export to Structured Format")
print("="*60)

# Create DataFrame
df = pd.DataFrame(extracted_companies)

# Add calculated fields
df['ebitda_margin'] = (df['ebitda'] / df['revenue'] * 100).round(1)
df['net_debt_to_ebitda'] = (df['net_debt'] / df['ebitda']).round(2)
df['extraction_date'] = datetime.now().strftime('%Y-%m-%d')
df['source_document'] = 'BlackRock Private Credit Fund II Q3 2024'

# Select output columns
output_columns = [
    'company_name', 'industry', 'revenue', 'ebitda', 'ebitda_margin',
    'net_debt', 'net_debt_to_ebitda', 'total_assets', 'total_equity',
    'debt_to_equity', 'extraction_date', 'source_document'
]
df_output = df[output_columns]

# Display
print("\n📋 Extracted Data Preview:")
df_output

In [ ]:
# Save to CSV and Excel
csv_path = 'extracted_portfolio_companies.csv'
excel_path = 'extracted_portfolio_companies.xlsx'

df_output.to_csv(csv_path, index=False)
df_output.to_excel(excel_path, index=False, sheet_name='Portfolio Companies')

print(f"\n💾 Files saved:")
print(f"   - {csv_path}")
print(f"   - {excel_path}")

# Download files in Colab
try:
    from google.colab import files
    print("\n📥 Download links:")
    files.download(csv_path)
    files.download(excel_path)
except:
    print("\n(Not running in Colab - files saved to current directory)")

## 4️⃣ (Optional) LLM API Integration

Uncomment and configure the code below to use actual LLM APIs for extraction.

In [ ]:
# =============================================================================
# OPENAI INTEGRATION
# =============================================================================

# import openai
# 
# # Set your API key
# openai.api_key = "your-openai-api-key"  # Or use: os.environ.get('OPENAI_API_KEY')
# 
# def extract_with_openai(text: str, target_metrics: List[str]) -> dict:
#     """Extract financial metrics using OpenAI GPT-4."""
#     prompt = LLMPromptBuilder.build_extraction_prompt(text, target_metrics)
#     
#     response = openai.ChatCompletion.create(
#         model="gpt-4",
#         messages=[
#             {"role": "system", "content": LLMPromptBuilder.SYSTEM_PROMPT},
#             {"role": "user", "content": prompt}
#         ],
#         temperature=0.1,  # Low temperature for factual extraction
#         response_format={"type": "json_object"}
#     )
#     
#     return json.loads(response.choices[0].message.content)

print("💡 OpenAI integration code ready - uncomment to use")

In [ ]:
# =============================================================================
# ANTHROPIC CLAUDE INTEGRATION
# =============================================================================

# import anthropic
# 
# # Set your API key
# client = anthropic.Anthropic(api_key="your-anthropic-api-key")
# 
# def extract_with_claude(text: str, target_metrics: List[str]) -> dict:
#     """Extract financial metrics using Anthropic Claude."""
#     prompt = LLMPromptBuilder.build_extraction_prompt(text, target_metrics)
#     
#     response = client.messages.create(
#         model="claude-3-sonnet-20240229",
#         max_tokens=4096,
#         messages=[{"role": "user", "content": prompt}]
#     )
#     
#     return json.loads(response.content[0].text)

print("💡 Anthropic Claude integration code ready - uncomment to use")

## 5️⃣ (Optional) Process Your Own PDF

Upload a PDF file and extract financial data from it.

In [ ]:
# Upload and process your own PDF
try:
    from google.colab import files
    print("📤 Upload a PDF file:")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        print(f"\n📄 Processing: {filename}")
        
        # Extract text and tables
        extractor = PDFTextExtractor(filename)
        full_text, tables = extractor.extract_all()
        
        print(f"\n📝 Extracted Text Preview (first 1000 chars):")
        print("-"*40)
        print(full_text[:1000])
        
        if tables:
            print(f"\n📊 Found {len(tables)} tables")
            print("First table preview:")
            display(tables[0].head())
except:
    print("(PDF upload only available in Google Colab)")

---

## ✅ Summary

This notebook demonstrates:

1. **PDF Text Extraction** - Using `pdfplumber` to extract text and tables from fund reports
2. **LLM Prompt Engineering** - Structured prompts for accurate financial metric extraction
3. **Data Validation** - Automated quality checks (balance sheet equation, net debt calculation)
4. **Structured Output** - Export to CSV/Excel for downstream analysis

### In Production:
- Replace simulated LLM response with actual API calls (OpenAI/Anthropic)
- Add batch processing for multiple fund reports
- Integrate with database for historical comparison
- Deploy as API service for automated data pipelines

---

**Author:** Arnab Banerjee  
**GitHub:** [github.com/arnab2308/financial-data-extractor](https://github.com/arnab2308/financial-data-extractor)